# Mimicry stress test: controls (A24)

`18` showed that replacing a few features of test Sybils collapses recall at the fixed threshold. This notebook
adds four controls, with the same model, threshold, donors and attacks as `18` (setup copied from `18`):

- **A. Single-feature sensitivity**: every feature replaced alone (k = 1).
- **B. Strict cheap**: a rule-based subset that does not depend on the tentative cost classes: Ethereum-side
  counts and values only (no LayerZero or bridging activity, no timing).
- **C. Structural only**: the gas-provider, gas-provision-tree and provision-chain features (excluding the
  time-bound `gas_provision_block_number`), up to all 30 replaced jointly.
- **D. Random-k**: k features chosen at random (20 subsets per k), from all 62 features and from the 25 `cheap`
  features, to separate fragility specific to the important features from general sensitivity to replacement.

Regression checks against `results/18` run first. The class names (`cheap`, `time-bound`, `structural`) are kept as
in `18`; they are tentative and for the authors to confirm.

**Results**: `results/24_mimicry_controls.json`, `figures/rev_mimicry_controls.{pdf,png}`,
`tables/rev_mimicry_controls.tex`, `tables/rev_single_feature_sensitivity.tex`.

## Setup (copied from `18`)

In [1]:
import os, platform, time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
PLATFORM = dict(machine=platform.machine(), processor=platform.processor(), system=platform.system(),
                python=platform.python_version(), cpu_count=os.cpu_count(), n_jobs=sp.N_JOBS)
print('Platform:', PLATFORM)
prov = sp.provenance(['09_shap_importance', '18_mimicry_stress'])
print(prov.to_string(index=False)); assert prov['dependencies_unchanged'].all()
DATA_DIR = './data'
FEATS = sp.FEATS
_, LGBM_P = sp.load_selected_params()

Code commit: 37bdba6
Platform: {'machine': 'x86_64', 'processor': 'x86_64', 'system': 'Linux', 'python': '3.11.15', 'cpu_count': 4, 'n_jobs': 4}


          notebook  commit  dependencies_unchanged
09_shap_importance 0d9eacc                    True
 18_mimicry_stress daa8986                    True


In [2]:
TIME_BOUND = ['l0_tx_time_span', 'earliest_l0_tx_time', 'latest_l0_tx_time', 'time_span_in', 'earliest_tx_block_in',
              'gas_provision_block_number', 'l0_to_eth_tx_time_span']
NETWORK = [f for k in ['Gas provider', 'Gas provision tree', 'Provision chain'] for f in sp.FEATURE_FAMILIES[k]]
STRUCTURAL = [f for f in NETWORK if f not in TIME_BOUND]
CHEAP = [f for f in FEATS if f not in TIME_BOUND and f not in STRUCTURAL]
FEATURE_COST = {**{f: 'cheap' for f in CHEAP}, **{f: 'time-bound' for f in TIME_BOUND}, **{f: 'structural' for f in STRUCTURAL}}
assert sorted(FEATURE_COST) == sorted(FEATS) and len(CHEAP) + len(TIME_BOUND) + len(STRUCTURAL) == len(FEATS)
print(f'cheap {len(CHEAP)}, time-bound {len(TIME_BOUND)}, structural {len(STRUCTURAL)}')
print('Cheap:', CHEAP)

shap = pd.DataFrame(sp.load_results('09_shap_importance')['mean_abs_shap'])
assert set(shap['feature']) == set(FEATS)
RANK = shap.sort_values('All', ascending=False)['feature'].tolist()
ORDERS = {'all features': RANK,
          'cheap only': [f for f in RANK if FEATURE_COST[f] == 'cheap'],
          'cheap + structural': [f for f in RANK if FEATURE_COST[f] in ('cheap', 'structural')]}
print(shap.sort_values('All', ascending=False).head(20).assign(cost=lambda t: t['feature'].map(FEATURE_COST))
      [['feature', 'All', 'family', 'cost']].round(4).to_string(index=False))

cheap 25, time-bound 7, structural 30
Cheap: ['min_tx_value_out', 'cex_in_count', 'l0_to_eth_avg_native_drop_usd', 'l0_to_eth_max_native_drop_usd', 'l0_avg_stargate_swap', 'indegree_per_block_in', 'l0_min_stargate_swap', 'num_transactions_in', 'n_l0_to_eth_source_contracts', 'n_l0_to_eth_projects', 'n_l0_to_eth_project_per_source_chain', 'n_l0_to_eth_txs', 'n_l0_projects', 'n_l0_to_eth_dest_contracts', 'l0_to_eth_min_stargate_swap', 'n_l0_source_chains', 'n_eth_interactions', 'max_tx_value_out', 'n_l0_source_contracts', 'min_tx_value_in', 'tx_value_per_block_out', 'n_l0_to_eth_source_chains', 'l0_to_eth_max_stargate_swap', 'n_l0_project_per_source_chain', 'l0_to_eth_avg_stargate_swap']
                          feature    All                 family       cost
                  l0_tx_time_span 0.9068 LayerZero transactions time-bound
            n_l0_source_contracts 0.7108 LayerZero transactions      cheap
             l0_avg_stargate_swap 0.6063 LayerZero transactions      cheap
     

In [3]:
df, _, _ = sp.build_master_df(DATA_DIR, verbose=False)
S = sp.make_splits(df, FEATS, method='group', seed=sp.SEED)
m = sp.fit_lgbm(S, LGBM_P)
THR = sp.best_f1_threshold(S['y_val'], m['val'])

def score(X):
    return np.mean([mod.predict_proba(X)[:, 1] for mod in m['models']], axis=0)

assert np.allclose(score(S['X_test']), m['test'], rtol=0, atol=1e-12)
X_te, y_te = S['X_test'].copy(), S['y_test'].to_numpy()
is_syb = y_te == 1
donors = df.loc[S['idx_train']]
donors = donors.loc[donors['sybil'] == 0, FEATS].astype(np.float32).to_numpy()   # unique non-Sybil training rows
print(f'Threshold {THR:.4f}; {is_syb.sum():,} test Sybils; {len(donors):,} donor rows')

def metrics(p):
    yhat = (p >= THR).astype(int)
    return dict(recall=float(yhat[is_syb].mean()), f1=f1_score(y_te, yhat), ap=average_precision_score(y_te, p),
                mean_sybil_score=float(p[is_syb].mean()))
BASE = metrics(m['test'])
print('Unattacked:', {k: round(v, 4) for k, v in BASE.items()})

  LightGBM seed=42  rounds=1277  88.6s


  LightGBM seed=123  rounds=1248  86.7s


  LightGBM seed=456  rounds=1143  76.4s


Threshold 0.6001; 5,463 test Sybils; 204,122 donor rows
Unattacked: {'recall': 0.7111, 'f1': 0.7188, 'ap': 0.7696, 'mean_sybil_score': 0.7188}


## Attack function and regression checks against `18`

In [4]:
import json, re
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
REPS = 5
col = {f: i for i, f in enumerate(FEATS)}          # as in 18
Xs = X_te.to_numpy()[is_syb]                       # as in 18

def attack_once(idx, attack, rng):
    A = Xs.copy()
    if attack == 'M1 marginal':
        for j in idx:
            A[:, j] = donors[rng.integers(0, len(donors), len(A)), j]
    else:
        d = rng.integers(0, len(donors), len(A))
        A[:, idx] = donors[d][:, idx]
    p = m['test'].copy()
    p[is_syb] = score(pd.DataFrame(A, columns=FEATS))
    return metrics(p)

def run(features, attack, seed_fn, reps=REPS):
    """Mean and SD over repetitions of attack_once on the given features; seed_fn(rep) -> rng seed."""
    idx = [col[f] for f in features]
    r = pd.DataFrame([attack_once(idx, attack, np.random.default_rng(seed_fn(rep))) for rep in range(reps)])
    return {f'{c}_{a}': float(getattr(r[c], a)()) for c in r.columns for a in ('mean', 'std')}

r18 = sp.load_results('18_mimicry_stress')
reg = dict(baseline={k: dict(here=float(BASE[k]), r18=float(r18['baseline'][k])) for k in ['recall', 'f1', 'ap']})
for k in ['recall', 'f1', 'ap']:
    assert abs(BASE[k] - r18['baseline'][k]) < 1e-9, ('baseline', k, BASE[k], r18['baseline'][k], PLATFORM)
s18 = pd.DataFrame(r18['results']).set_index(['features', 'attack', 'k'])
reg['recall'] = []
for order in ['all features', 'cheap only']:
    for attack in ['M1 marginal', 'M2 profile copy']:
        for k in [1, 5, 10]:
            here = run(ORDERS[order][:k], attack, lambda rep, k=k: 1000 * rep + k)['recall_mean']
            there = float(s18.loc[(order, attack, k), 'recall_mean'])
            reg['recall'].append(dict(order=order, attack=attack, k=k, here=here, r18=there, diff=here - there))
            assert abs(here - there) < 1e-9, (order, attack, k, here, there, PLATFORM)
print(f"Regression vs 18: baseline (3 metrics) and {len(reg['recall'])} recall checks pass")

Regression vs 18: baseline (3 metrics) and 12 recall checks pass


## A. Single-feature sensitivity (k = 1, every feature; M2 equals M1 at k = 1)

In [5]:
t0 = time.time()
shap_rank = {f: i + 1 for i, f in enumerate(RANK)}
single = []
for f in FEATS:
    r = run([f], 'M1 marginal', lambda rep: 1000 * rep + 1)
    single.append(dict(component='A single feature', feature=f, cost_class=FEATURE_COST[f], shap_rank=shap_rank[f], **r))
single = pd.DataFrame(single).sort_values('recall_mean').reset_index(drop=True)
print(f'({time.time() - t0:.0f}s)')
print(single[['feature', 'cost_class', 'shap_rank', 'recall_mean', 'recall_std', 'f1_mean', 'ap_mean']].head(15).round(4).to_string(index=False))
a = single.set_index('feature').loc['l0_tx_time_span', 'recall_mean']
b = single.set_index('feature').loc['n_l0_source_contracts', 'recall_mean']
print(f'\nrecall l0_tx_time_span = {a:.6f}; n_l0_source_contracts = {b:.6f}; equal at 6 decimals: {round(a, 6) == round(b, 6)}')
# If equal: check that the two columns, and the donor values drawn for them, really differ
ja, jb = col['l0_tx_time_span'], col['n_l0_source_contracts']
rng_a, rng_b = np.random.default_rng(1), np.random.default_rng(1)    # rep 0 seed for k = 1
da = donors[rng_a.integers(0, len(donors), len(Xs)), ja]; db = donors[rng_b.integers(0, len(donors), len(Xs)), jb]
pa = m['test'].copy(); A = Xs.copy(); A[:, ja] = da; pa[is_syb] = score(pd.DataFrame(A, columns=FEATS))
pb = m['test'].copy(); B = Xs.copy(); B[:, jb] = db; pb[is_syb] = score(pd.DataFrame(B, columns=FEATS))
tie_check = dict(recall_l0_tx_time_span=float(a), recall_n_l0_source_contracts=float(b), equal_6dp=bool(round(a, 6) == round(b, 6)),
                 columns_differ=bool(not np.array_equal(Xs[:, ja], Xs[:, jb])), donor_values_differ=bool(not np.array_equal(da, db)),
                 same_donor_rows=True, sybil_scores_differ_rep0=bool(not np.array_equal(pa[is_syb], pb[is_syb])),
                 flagged_rep0=[int((pa[is_syb] >= THR).sum()), int((pb[is_syb] >= THR).sum())],
                 flagged_both_rep0=int(((pa[is_syb] >= THR) & (pb[is_syb] >= THR)).sum()))
print(tie_check)

(381s)
                      feature cost_class  shap_rank  recall_mean  recall_std  f1_mean  ap_mean
              l0_tx_time_span time-bound          1       0.4098      0.0059   0.4886   0.5163
        n_l0_source_contracts      cheap          2       0.4125      0.0031   0.4911   0.5141
          earliest_l0_tx_time time-bound          4       0.4219      0.0032   0.4995   0.5193
         l0_avg_stargate_swap      cheap          3       0.4744      0.0024   0.5447   0.5687
         earliest_tx_block_in time-bound         10       0.5174      0.0021   0.5797   0.5983
           n_l0_source_chains      cheap          8       0.5371      0.0030   0.5953   0.6181
            latest_l0_tx_time time-bound          5       0.5446      0.0050   0.6011   0.6148
          num_transactions_in      cheap          7       0.5487      0.0028   0.6043   0.6404
             max_tx_value_out      cheap          6       0.5707      0.0029   0.6209   0.6370
                 time_span_in time-bound   

{'recall_l0_tx_time_span': 0.4097748489840747, 'recall_n_l0_source_contracts': 0.41252059308072486, 'equal_6dp': False, 'columns_differ': True, 'donor_values_differ': True, 'same_donor_rows': True, 'sybil_scores_differ_rep0': True, 'flagged_rep0': [2218, 2267], 'flagged_both_rep0': 1727}


## B. Strict-cheap subset

In [6]:
# Strict cheap = Ethereum-side counts and values only: no LayerZero or bridging activity, no timing
STRICT_CHEAP = [f for f in sp.FEATURE_FAMILIES['Ethereum transactions'] if f not in TIME_BOUND]
assert len(STRICT_CHEAP) == 7 and set(STRICT_CHEAP) <= set(CHEAP)
ORDER_STRICT = [f for f in RANK if f in STRICT_CHEAP]
# expected: max_tx_value_out, num_transactions_in, tx_value_per_block_out, min_tx_value_in,
#           min_tx_value_out, cex_in_count, indegree_per_block_in
print('Strict-cheap order:', ORDER_STRICT)
strict = []
for k in [1, 2, 3, 5, 7]:
    for attack in ['M1 marginal', 'M2 profile copy']:
        strict.append(dict(component='B strict cheap', k=k, attack=attack, features=ORDER_STRICT[:k],
                           **run(ORDER_STRICT[:k], attack, lambda rep, k=k: 1000 * rep + k)))
strict = pd.DataFrame(strict)
print(strict.pivot(index='k', columns='attack', values='recall_mean').round(4).to_string())

Strict-cheap order: ['max_tx_value_out', 'num_transactions_in', 'tx_value_per_block_out', 'min_tx_value_in', 'min_tx_value_out', 'cex_in_count', 'indegree_per_block_in']


attack  M1 marginal  M2 profile copy
k                                   
1            0.5707           0.5707
2            0.3643           0.3640
3            0.2541           0.2605
5            0.1605           0.1627
7            0.1370           0.1404


## C. Structural only

In [7]:
ORDER_STRUCT = [f for f in RANK if FEATURE_COST[f] == 'structural']
assert len(ORDER_STRUCT) == 30
struct = []
for k in [1, 2, 3, 5, 10, 20, 30]:
    for attack in ['M1 marginal', 'M2 profile copy']:
        struct.append(dict(component='C structural only', k=k, attack=attack, features=ORDER_STRUCT[:k],
                           label='all 30 structural features' if k == 30 else f'top {k} structural',
                           **run(ORDER_STRUCT[:k], attack, lambda rep, k=k: 1000 * rep + k)))
struct = pd.DataFrame(struct)
print('Structural order (top 5):', ORDER_STRUCT[:5])
print(struct.pivot(index='k', columns='attack', values='recall_mean').round(4).to_string())

Structural order (top 5): ['provider_avg_gas_provision_amount', 'provider_min_gas_provision_amount', 'provider_fan_out', 'provider_total_gas_provision_amount', 'provider_max_gas_provision_amount']
attack  M1 marginal  M2 profile copy
k                                   
1            0.6903           0.6903
2            0.6547           0.6537
3            0.6083           0.6059
5            0.5238           0.5314
10           0.4832           0.4960
20           0.4653           0.4712
30           0.4566           0.4685


## D. Random-k control (20 subsets per k and pool)

In [8]:
t0 = time.time()
POOLS = {'all features': list(FEATS), 'cheap': list(CHEAP)}
rk_rows, subsets = [], {}
for pool, feats in POOLS.items():
    for k in [1, 2, 3, 5, 10, 20]:
        for rep in range(20):
            sub = list(np.random.default_rng(10_000 + 100 * k + rep).choice(feats, k, replace=False))
            subsets[f'{pool}|{k}|{rep}'] = sub
            idx = [col[f] for f in sub]
            for attack in ['M1 marginal', 'M2 profile copy']:
                r = attack_once(idx, attack, np.random.default_rng(50_000 + 100 * k + rep))
                rk_rows.append(dict(pool=pool, k=k, subset=rep, attack=attack, **r))
rk = pd.DataFrame(rk_rows)
random_k = (rk.groupby(['pool', 'k', 'attack'])['recall']
            .agg(recall_mean='mean', recall_std='std', recall_min='min', recall_max='max', subsets='size').reset_index())
random_k['component'] = 'D random-k'
print(f'({time.time() - t0:.0f}s)')
print(random_k.round(4).to_string(index=False))

(600s)
        pool  k          attack  recall_mean  recall_std  recall_min  recall_max  subsets  component
all features  1     M1 marginal       0.6736      0.0753      0.4227      0.7111       20 D random-k
all features  1 M2 profile copy       0.6736      0.0753      0.4227      0.7111       20 D random-k
all features  2     M1 marginal       0.6017      0.1318      0.2618      0.7100       20 D random-k
all features  2 M2 profile copy       0.6022      0.1312      0.2682      0.7102       20 D random-k
all features  3     M1 marginal       0.5697      0.1647      0.1583      0.7095       20 D random-k
all features  3 M2 profile copy       0.5724      0.1588      0.1906      0.7102       20 D random-k
all features  5     M1 marginal       0.4186      0.1769      0.1294      0.7110       20 D random-k
all features  5 M2 profile copy       0.4238      0.1691      0.1693      0.7100       20 D random-k
all features 10     M1 marginal       0.2350      0.2023      0.0229      0.6370    

## SHAP shares and the SHAP-ordered curves for the figure

In [9]:
sh = pd.DataFrame(sp.load_results('09_shap_importance')['mean_abs_shap']).set_index('feature')['All']
tot = sh.sum()
share = lambda fs: float(sh.loc[list(fs)].sum() / tot)
shap_shares = {'top5_all_features': share(ORDERS['all features'][:5]), 'top5_cheap_only': share(ORDERS['cheap only'][:5]),
               'top5_strict_cheap': share(ORDER_STRICT[:5]), 'all7_strict_cheap': share(ORDER_STRICT),
               'all30_structural': share(ORDER_STRUCT)}
print({k: round(v, 4) for k, v in shap_shares.items()})
shap_curves = []
for order in ['all features', 'cheap only']:
    for k in [1, 2, 3, 5, 10, 20]:
        for attack in ['M1 marginal', 'M2 profile copy']:
            shap_curves.append(dict(component='SHAP order (as in 18)', order=order, k=k, attack=attack,
                                    **run(ORDERS[order][:k], attack, lambda rep, k=k: 1000 * rep + k)))
shap_curves = pd.DataFrame(shap_curves)

{'top5_all_features': 0.3767, 'top5_cheap_only': 0.3116, 'top5_strict_cheap': 0.1939, 'all7_strict_cheap': 0.2148, 'all30_structural': 0.1324}


## Figure `rev_mimicry_controls`

In [10]:
C_XGB, C_SHARED, C_RF, C_ENS = '#1565C0', '#E65100', '#455A64', '#6A1B9A'
# Display names agreed by the authors; the internal class keys ('cheap', 'time-bound', 'structural') are unchanged
DISPLAY = {'cheap': 'wallet-local', 'time-bound': 'time-based', 'structural': 'funding-graph'}
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
SMALL = 8.5
os.makedirs('figures', exist_ok=True); os.makedirs('tables', exist_ok=True)
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.5, 3.4), gridspec_kw=dict(width_ratios=[1, 1]))
M2 = 'M2 profile copy'
def curve(df, label, color, ls, mk, kcol='k'):
    t = df.sort_values(kcol)
    ax.errorbar(t[kcol], t['recall_mean'], yerr=t['recall_std'], color=color, ls=ls, marker=mk, ms=4, lw=1.2, capsize=2, label=label)
curve(shap_curves[(shap_curves.order == 'all features') & (shap_curves.attack == M2)], 'SHAP order, all features', C_XGB, '-', 'o')
curve(shap_curves[(shap_curves.order == 'cheap only') & (shap_curves.attack == M2)], 'SHAP order, wallet-local only', C_SHARED, '--', 's')
curve(strict[strict.attack == M2], 'Ethereum-local (7)', C_ENS, '-.', '^')
curve(struct[struct.attack == M2], 'funding-graph only (30)', C_RF, ':', 'D')
ra = random_k[(random_k.pool == 'all features') & (random_k.attack == M2)].sort_values('k')
ax.fill_between(ra['k'], ra['recall_min'], ra['recall_max'], color='0.75', alpha=0.5, lw=0)
ax.plot(ra['k'], ra['recall_mean'], color='0.35', lw=1.2, ls=(0, (5, 1, 1, 1)), marker='x', ms=5, label='random k of 62')
ax.axhline(BASE['recall'], color='0.2', lw=0.9, ls=':', label=None)
ax.text(1.02, BASE['recall'] + 0.015, f"no attack ({BASE['recall']:.3f})", fontsize=SMALL, color='0.2')
ax.set_xscale('log'); ax.set_xticks([1, 2, 3, 5, 10, 20, 30]); ax.set_xticklabels([1, 2, 3, 5, 10, 20, 30], fontsize=SMALL + 0.5)
ax.set_ylim(-0.02, 0.85); ax.set_xlabel('Features replaced (k), profile copy (M2)', fontsize=10); ax.set_ylabel('Recall of test Sybils\n(fixed threshold)')
ax.legend(fontsize=SMALL - 1, frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2)
ax.set_title('A  Attack scope', loc='left', fontsize=11)
top = single.head(15).iloc[::-1]
TAG = {'cheap': 'L', 'time-bound': 'T', 'structural': 'G'}
CC = {'cheap': C_SHARED, 'time-bound': C_XGB, 'structural': C_RF}
HATCH = {'cheap': '', 'time-bound': '///', 'structural': '...'}
y = np.arange(len(top))
for yi, (_, r) in zip(y, top.iterrows()):
    bx.barh(yi, r['recall_mean'], xerr=r['recall_std'], color=CC[r['cost_class']], hatch=HATCH[r['cost_class']], edgecolor='white',
            lw=0, error_kw=dict(lw=0.8, capsize=1.5))
bx.axvline(BASE['recall'], color='0.2', lw=0.9, ls=':')
bx.set_yticks(y); bx.set_yticklabels([f"{r['feature']} [{TAG[r['cost_class']]}]" for _, r in top.iterrows()], fontsize=SMALL - 0.5)
bx.set_xlim(0, 0.8); bx.set_xlabel('Recall, one feature replaced', fontsize=10)
from matplotlib.patches import Patch
bx.legend(handles=[Patch(facecolor=CC[c], hatch=HATCH[c], edgecolor='white', label=f'{DISPLAY[c]} [{TAG[c]}]') for c in CC] +
          [Line2D([], [], color='0.2', ls=':', label='no attack')], fontsize=SMALL - 1, frameon=False, loc='upper center',
          bbox_to_anchor=(0.35, -0.24), ncol=2)
bx.set_title('B  15 most sensitive single features', loc='left', fontsize=11)
fig.tight_layout()
for ext, kw in [('pdf', {}), ('png', dict(dpi=200))]:
    fig.savefig(f'figures/rev_mimicry_controls.{ext}', bbox_inches='tight', **kw)
plt.close(fig)
FIG_FILES = ['figures/rev_mimicry_controls.pdf', 'figures/rev_mimicry_controls.png']
assert all(os.path.getsize(f) > 0 for f in FIG_FILES)
print('Saved figures/rev_mimicry_controls.{pdf,png}')

Saved figures/rev_mimicry_controls.{pdf,png}


## Tables

In [11]:
pm = lambda mu, sd: f'{mu:.3f} $\\pm$ {sd:.3f}'
KS = [1, 3, 5, 7, 10]   # k = 7 exists only for strict cheap (7 features)
def scope_rows(attack):
    rows = {}
    for order, lab in [('all features', 'All features (SHAP order)'), ('cheap only', 'Wallet-local only (SHAP order)')]:
        t = shap_curves[(shap_curves.order == order) & (shap_curves.attack == attack)].set_index('k')
        rows[lab] = [pm(t.loc[k, 'recall_mean'], t.loc[k, 'recall_std']) if k in t.index else '--' for k in KS]
    t = strict[strict.attack == attack].set_index('k')
    rows['Ethereum-local only (SHAP order)'] = [pm(t.loc[k, 'recall_mean'], t.loc[k, 'recall_std']) if k in t.index else '--' for k in KS]
    t = struct[struct.attack == attack].set_index('k')
    rows['Funding-graph only (SHAP order)'] = [pm(t.loc[k, 'recall_mean'], t.loc[k, 'recall_std']) if k in t.index else '--' for k in KS]
    for pool, lab in [('all features', 'Random $k$ of all 62'), ('cheap', 'Random $k$ of wallet-local (25)')]:
        t = random_k[(random_k.pool == pool) & (random_k.attack == attack)].set_index('k')
        rows[lab] = [pm(t.loc[k, 'recall_mean'], t.loc[k, 'recall_std']) if k in t.index else '--' for k in KS]
    return rows
EXPECT = {a: scope_rows(a) for a in ['M1 marginal', 'M2 profile copy']}
lines = ['% Recall of test Sybils at the fixed validation threshold, mean $\\pm$ SD (SHAP-ordered scopes: over 5 repetitions; random k: over 20 subsets). Ethereum-local has 7 features.',
         f'% Generated by 24_mimicry_controls.ipynb at {sp.CODE_COMMIT} from its own results (results/24_mimicry_controls.json)',
         '\\begin{tabular}{l' + 'r' * len(KS) + '}', '\\toprule', 'Attack scope & ' + ' & '.join(f'$k={k}$' for k in KS) + ' \\\\', '\\midrule',
         f"No attack & \\multicolumn{{{len(KS)}}}{{r}}{{{BASE['recall']:.3f}}} \\\\"]
for attack, lab in [('M1 marginal', 'M1 marginal'), ('M2 profile copy', 'M2 profile copy')]:
    lines += ['\\midrule', f'\\multicolumn{{{len(KS) + 1}}}{{l}}{{\\emph{{{lab}}}}} \\\\']
    lines += [f'{name} & ' + ' & '.join(cells) + ' \\\\' for name, cells in EXPECT[attack].items()]
lines += ['\\bottomrule', '\\end{tabular}', '']
T_CTRL = '\n'.join(lines)
open('tables/rev_mimicry_controls.tex', 'w').write(T_CTRL)
print(T_CTRL)
top15 = single.head(15)
lines = ['% The 15 features whose replacement alone (M1, k = 1, 5 repetitions) lowers test-Sybil recall most; class tentative (authors to confirm).',
         f'% Generated by 24_mimicry_controls.ipynb at {sp.CODE_COMMIT}',
         '\\begin{tabular}{llrr}', '\\toprule', 'Feature & Class & SHAP rank & Recall \\\\', '\\midrule']
tex_escape = lambda s: s.replace('_', r'\_')   # outside the f-string: Python < 3.12 forbids backslashes in f-string expressions
lines += [f"\\texttt{{{tex_escape(r['feature'])}}} & {DISPLAY[r['cost_class']]} & {int(r['shap_rank'])} & {pm(r['recall_mean'], r['recall_std'])} \\\\"
          for _, r in top15.iterrows()]
lines += ['\\midrule', f"No attack & & & {BASE['recall']:.3f} \\\\", '\\bottomrule', '\\end{tabular}', '']
T_SINGLE = '\n'.join(lines)
open('tables/rev_single_feature_sensitivity.tex', 'w').write(T_SINGLE)
print(T_SINGLE)

% Recall of test Sybils at the fixed validation threshold, mean $\pm$ SD (SHAP-ordered scopes: over 5 repetitions; random k: over 20 subsets). Ethereum-local has 7 features.
% Generated by 24_mimicry_controls.ipynb at 37bdba6 from its own results (results/24_mimicry_controls.json)
\begin{tabular}{lrrrrr}
\toprule
Attack scope & $k=1$ & $k=3$ & $k=5$ & $k=7$ & $k=10$ \\
\midrule
No attack & \multicolumn{5}{r}{0.711} \\
\midrule
\multicolumn{6}{l}{\emph{M1 marginal}} \\
All features (SHAP order) & 0.410 $\pm$ 0.006 & 0.074 $\pm$ 0.002 & 0.012 $\pm$ 0.001 & -- & 0.000 $\pm$ 0.000 \\
Wallet-local only (SHAP order) & 0.413 $\pm$ 0.003 & 0.113 $\pm$ 0.002 & 0.025 $\pm$ 0.002 & -- & 0.003 $\pm$ 0.001 \\
Ethereum-local only (SHAP order) & 0.571 $\pm$ 0.003 & 0.254 $\pm$ 0.004 & 0.160 $\pm$ 0.004 & 0.137 $\pm$ 0.007 & -- \\
Funding-graph only (SHAP order) & 0.690 $\pm$ 0.000 & 0.608 $\pm$ 0.003 & 0.524 $\pm$ 0.001 & -- & 0.483 $\pm$ 0.003 \\
Random $k$ of all 62 & 0.674 $\pm$ 0.075 & 0.570 $\pm

## Checks

In [12]:
assert sorted(FEATURE_COST) == sorted(FEATS) and set(STRICT_CHEAP) <= set(CHEAP)
# rev_mimicry_controls.tex equals the results after rounding (each row's cells)
for attack, rows in EXPECT.items():
    block = T_CTRL.split(f'\\emph{{{attack}}}')[1].split('\\midrule')[0] if attack == 'M1 marginal' else T_CTRL.split(f'\\emph{{{attack}}}')[1]
    for name, cells in rows.items():
        hit = [l for l in block.splitlines() if l.startswith(name + ' &')]
        assert len(hit) == 1 and [x.strip() for x in hit[0].rstrip(' \\').split(' & ')[1:]] == cells, (attack, name)
for name, text in [('rev_mimicry_controls', T_CTRL), ('rev_single_feature_sensitivity', T_SINGLE)]:
    ncol = len(re.search(r'\\begin\{tabular\}\{([a-z]+)\}', text).group(1))
    for l in text.splitlines():
        if l.endswith('\\\\'):
            spans = sum(int(s) - 1 for s in re.findall(r'\\multicolumn\{(\d+)\}', l))
            assert len(re.findall(r'(?<!\\)&', l)) + 1 + spans == ncol, (name, l)   # unescaped & separators
import shutil, subprocess, tempfile
if shutil.which('pdflatex'):
    with tempfile.TemporaryDirectory() as d:
        body = ''.join(f'\\input{{{os.path.abspath(f)}}}\n\\bigskip\n' for f in ['tables/rev_mimicry_controls.tex', 'tables/rev_single_feature_sensitivity.tex'])
        open(os.path.join(d, 'w.tex'), 'w').write('\\documentclass{article}\n\\usepackage{booktabs}\n\\begin{document}\n' + body + '\\end{document}\n')
        assert subprocess.run(['pdflatex', '-interaction=nonstopmode', 'w.tex'], cwd=d, capture_output=True).returncode == 0
    COMPILE = 'pdflatex: tables compile'
else:
    COMPILE = 'WARNING: pdflatex not installed; compile check skipped (structural column check passed)'
print(COMPILE)

pdflatex: tables compile


## Save results

In [13]:
recs = lambda df: json.loads(df.to_json(orient='records'))
# Reproducibility check against the previous committed result of 24. Its numbers must equal these to 1e-9 when it was
# produced on this platform from the same effective inputs: sybil_pipeline.py, requirements.txt, data/, the code cells
# of this notebook, the LightGBM hyperparameters selected by 02 (the only ones 24 uses), and the results and code cells
# of 09 (SHAP order) and 18 (regression reference). When any of them changed, or the platform differs, differences are
# expected: they are computed and recorded with the inputs that changed, not asserted. The checks against the current
# run (provenance of 09 and 18, the regression checks against 18, the feature and table checks) do not depend on this.
import subprocess, nbformat
prev = sp.load_results('24_mimicry_controls')
PREV = prev['code_commit']
same_platform = prev.get('platform', {}).get('machine') == PLATFORM['machine'] and prev.get('platform', {}).get('system') == PLATFORM['system']
git = lambda *a: subprocess.run(['git', *a], capture_output=True, text=True)
def at(ref, path):                       # a committed file's text; None when the commit or the file is not available
    r = git('show', f'{ref}:{path}')
    return r.stdout if r.returncode == 0 else None
def code_cells(text):
    return None if text is None else [c.source for c in nbformat.reads(text, as_version=4).cells if c.cell_type == 'code']
def unstamped(r):                        # a result without its commit stamp and host description (24's own platform is checked above)
    return None if r is None else {k: v for k, v in r.items() if k not in ('code_commit', 'platform')}
INPUTS = ['sybil_pipeline.py', 'requirements.txt', 'data/', '24_mimicry_controls code cells', '02 selected LightGBM hyperparameters',
          '09_shap_importance result', '09_shap_importance code cells', '18_mimicry_stress result', '18_mimicry_stress code cells']
if PREV.endswith('-dirty') or git('cat-file', '-e', f'{PREV}^{{commit}}').returncode != 0:
    changed_inputs = [f'all (previous commit {PREV} is not a clean commit of this repository)']
else:                                    # previous commit against the commit this kernel runs (clean, asserted at the top)
    changed_inputs = [p for p in INPUTS[:3] if git('diff', '--quiet', PREV, sp.CODE_COMMIT, '--', p.rstrip('/')).returncode != 0]
    prev_02 = json.loads(at(PREV, 'results/02_hyperparameter_search.json') or '{}')
    if prev_02.get('lgbm_selected') != LGBM_P or prev.get('params') != LGBM_P:
        changed_inputs.append('02 selected LightGBM hyperparameters')
    for nb in ['09_shap_importance', '18_mimicry_stress']:
        old = at(PREV, f'results/{nb}.json')
        if unstamped(None if old is None else json.loads(old)) != unstamped(sp.load_results(nb)):
            changed_inputs.append(f'{nb} result')
    for nb in ['24_mimicry_controls', '09_shap_importance', '18_mimicry_stress']:
        if code_cells(at(PREV, f'{nb}.ipynb')) != code_cells(at(sp.CODE_COMMIT, f'{nb}.ipynb')):
            changed_inputs.append(f'{nb} code cells')
comparison_asserted = bool(same_platform and not changed_inputs)
unmatched = {}                           # not asserted: rows or keys found in only one of the two results
def max_diff(name, new, old, keys, num):
    a = pd.DataFrame(new).set_index(keys)[num]
    if comparison_asserted:
        b = pd.DataFrame(old).set_index(keys)[num]
        assert a.index.sort_values().equals(b.index.sort_values())
        return float((a - b.loc[a.index]).abs().max().max())
    missing = sorted(set(keys + num) - set().union(*(r.keys() for r in old))) if old else []
    b = pd.DataFrame(old or [], columns=keys + num).set_index(keys)[num]
    common = a.index.intersection(b.index)
    if len(common) < max(len(a), len(b)):
        unmatched[name] = dict(only_new=int(len(a.index.difference(b.index))), only_previous=int(len(b.index.difference(a.index))))
    if missing:                          # a column absent from the previous result is reported, not counted as a 0 difference
        unmatched.setdefault(name, {})['columns_missing_previous'] = missing
    return float((a.loc[common] - b.loc[common]).abs().max().max()) if len(common) else None
def scalar_diff(name, new, old):
    if comparison_asserted:
        return max(abs(new[k] - old[k]) for k in new)
    old = old or {}
    if set(new) != set(old):
        unmatched[name] = dict(only_new=len(set(new) - set(old)), only_previous=len(set(old) - set(new)))
    return max((abs(new[k] - old[k]) for k in new if k in old), default=None)
NUMS = ['recall_mean', 'recall_std', 'f1_mean', 'ap_mean']
diffs = dict(single_feature=max_diff('single_feature', recs(single), prev.get('single_feature'), ['feature'], NUMS),
             strict=max_diff('strict', recs(strict), prev.get('strict_cheap_results'), ['k', 'attack'], NUMS),
             structural=max_diff('structural', recs(struct), prev.get('structural_results'), ['k', 'attack'], NUMS),
             random_k=max_diff('random_k', recs(random_k), prev.get('random_k_results'), ['pool', 'k', 'attack'],
                               ['recall_mean', 'recall_std', 'recall_min', 'recall_max']),
             shap_ordered=max_diff('shap_ordered', recs(shap_curves), prev.get('shap_ordered_results'), ['order', 'k', 'attack'], NUMS),
             baseline=scalar_diff('baseline', {k: BASE[k] for k in ['recall', 'f1', 'ap']},
                                  {k: v for k, v in (prev.get('baseline') or {}).items() if k in ('recall', 'f1', 'ap')}),
             shap_shares=scalar_diff('shap_shares', shap_shares, prev.get('shap_shares')))
PREV_CHECK = dict(previous_commit=PREV, same_platform=same_platform, inputs_compared=INPUTS, changed_inputs=changed_inputs,
                  comparison_asserted=comparison_asserted, max_abs_diff=diffs, unmatched=unmatched)
print(PREV_CHECK)
if comparison_asserted:
    assert all(v < 1e-9 for v in diffs.values()), diffs
else:
    why = ([] if same_platform else ['platform']) + changed_inputs
    print(f"NOTE: previous result of 24 ({PREV}) differs in {'; '.join(why)}; numbers compared and recorded, not asserted.")
sp.save_results([], '24_mimicry_controls', extra=dict(
    previous_result_check=PREV_CHECK, display_names=DISPLAY,
    params=LGBM_P, threshold=THR, baseline=BASE, ks=dict(single=[1], strict=[1, 2, 3, 5, 7], structural=[1, 2, 3, 5, 10, 20, 30],
                                                         random_k=[1, 2, 3, 5, 10, 20]),
    repetitions=REPS, random_k_subsets_per_k=20, regression_check=reg,
    feature_cost=FEATURE_COST, feature_cost_status='tentative, for the authors to confirm', strict_cheap=STRICT_CHEAP,
    strict_cheap_rule='Ethereum transactions family minus time-bound features: Ethereum-side counts and values only',
    orders={'all features': ORDERS['all features'], 'cheap only': ORDERS['cheap only'], 'strict cheap': ORDER_STRICT,
            'structural': ORDER_STRUCT},
    single_feature=recs(single), single_feature_tie_check=tie_check, strict_cheap_results=recs(strict),
    structural_results=recs(struct), random_k_results=recs(random_k), random_k_per_subset=recs(rk), random_k_subsets=subsets,
    shap_ordered_results=recs(shap_curves), shap_shares=shap_shares,
    figures=FIG_FILES, tables=['tables/rev_mimicry_controls.tex', 'tables/rev_single_feature_sensitivity.tex'], compile_check=COMPILE,
    caveats=r18['caveats'] + ['The random-k control is not an attack model; it shows general sensitivity to feature replacement.',
                              'Strict-cheap is defined by data source, not by monetary cost.'],
    platform=PLATFORM))

{'previous_commit': '2f39463', 'same_platform': False, 'max_abs_diff': {'single_feature': 0.022954420699999978, 'strict': 0.013618890699999997, 'structural': 0.016511074499999945, 'random_k': 0.014277869300000023, 'shap_ordered': 0.009957898600000015, 'baseline': 0.007505033864177224, 'shap_shares': 0.0}}
NOTE: previous result of 24 is from another platform; numbers compared but not asserted.
Saved results/24_mimicry_controls.json
